# Causal feature-engineered CNN + LSTM
An opt-in experiment based on `training_lstm.ipynb`: the same causal CNN, LSTM, classifier, delayed targets, positive-window augmentation, loss, AdamW optimizer, cosine schedule, and window-based checkpoint selection. The input projection receives engineered features instead of just the six IMU axes. Training starts from scratch.

The model still accepts `[batch, time, 6]` in the order `acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z`. Features are computed **inside the model**, after the dataset's high-pass filter and any raw-axis amplitude augmentation. No features use labels, future frames, validation statistics, or whole-recording normalization.

Open with the repository `.venv` kernel and run in order. Every training invocation creates a new directory under `testing_checkpoints/lstm64_features/`. The data and existing model runs are read-only inputs.

In [17]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'tap_recognition').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'tap_recognition').is_dir(), 'Open from repository root or main_notebooks/'
sys.path.insert(0, str(ROOT))
from tap_recognition import model_factory
importlib.reload(model_factory)
from tap_recognition.config import DataConfig, EarlyStoppingConfig, LSTMModelConfig, TrainConfig, TrainingConfig
from tap_recognition.imu_features import CausalIMUFeatures, IMUFeatureConfig
from tap_recognition.feature_training import build_feature_datasets, train_feature_lstm
torch.set_num_threads(2)


## Experiment settings
Keep seed, delay, split, augmentation, and optimizer settings matched to your raw-axis baseline. The current defaults use LSTM64, one layer, delay 10, seed 101, and the expanded split. Choose seed 102 in a separate invocation to examine training-seed variation.

Default **20 channels**: six signed axes; acceleration/gyro magnitudes; six per-sample differences; two difference magnitudes; and acceleration/gyro trailing RMS at 5 and 15 samples (50/150 ms at 100 Hz). Signed axes preserve left/right information. RMS is the square root of trailing mean squared vector magnitude, with zero prehistory. The first difference at a reset is zero. Disable groups or change `rms_windows` for ablations.

Training-only per-channel mean/std normalization gives the extra channels comparable scales. Statistics are fitted once on the unaugmented training windows (including their existing padding), frozen afterwards, and saved with the model weights.

In [18]:
DATA_ROOT = ROOT / 'data_09_10_26'
TRAINING_SEED = 46
LABEL_DELAY_FRAMES = 8
AMPLITUDE_AUGMENTATION = True
FEATURES = IMUFeatureConfig(
    magnitudes=True,
    differences=True,
    difference_magnitudes=True,
    rms_windows=(5, 15),
)
cfg = TrainConfig(
    seed=TRAINING_SEED,
    device='auto',
    out_dir=str(ROOT / 'testing_checkpoints/lstm64_features'),
    data=DataConfig(
        train_dir=str(DATA_ROOT / 'train_data'),
        val_dir=str(DATA_ROOT / 'valid_data'),
        session_exclusions_file=str(DATA_ROOT / 'session_exclusions.csv'),
        highpass=True,
    ),
    model=LSTMModelConfig(lstm_hidden=64, lstm_layers=1, dilations=(1, 2, 4, 8)),
    training=TrainingConfig(
        epochs=60, batch_size=16, lr=1e-3,
        early_stopping=EarlyStoppingConfig(monitor='val_window_acc'),
    ),
)
for name in ('train_data', 'valid_data'):
    if not (DATA_ROOT / name).is_dir():
        raise FileNotFoundError(DATA_ROOT / name)
if not Path(cfg.data.session_exclusions_file).is_file():
    raise FileNotFoundError(cfg.data.session_exclusions_file)
print(json.dumps(cfg.to_dict(), indent=2))
display(pd.DataFrame({'index': range(len(FEATURES.names)), 'feature': FEATURES.names}))


{
  "seed": 46,
  "device": "auto",
  "out_dir": "/home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/testing_checkpoints/lstm64_features",
  "init_checkpoint": null,
  "freeze_except_last": false,
  "data": {
    "mode": "recorded",
    "train_dir": "/home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_09_10_26/train_data",
    "val_dir": "/home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_09_10_26/valid_data",
    "window_samples": 300,
    "sample_rate": 100.0,
    "highpass": true,
    "negative_windows_per_file": 20,
    "trigger_context_samples": 200,
    "exclusion_margin": 200,
    "dt_min": 0.12,
    "dt_max": 0.45,
    "session_exclusions_file": "/home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_09_10_26/session_exclusions.csv",
    "synthetic_samples": 8000,
    "val_split": 0.15
  },
  "labels": {
    "sigma": 0.04,
    "peak_offset": 0.05,
  

,index,feature
0,0,acc_x
1,1,acc_y
2,2,acc_z
3,3,gyro_x
4,4,gyro_y
5,5,gyro_z
6,6,acc_magnitude
7,7,gyro_magnitude
8,8,delta_acc_x
9,9,delta_acc_y


## Load and inspect the explicit split
This uses the same recorded-window loader as the baseline and the selected split's exclusion registry. An invalid validation split is reported rather than replaced by a random subset of training. No output directory has been created yet.

In [19]:
train_ds, val_ds = build_feature_datasets(cfg)
balance = []
for split, dataset in [('train', train_ds), ('valid', val_ds)]:
    counts = pd.Series([int(item['window_label']) for item in dataset]).value_counts()
    balance.append({'split': split, 'windows': len(dataset),
                    'none': int(counts.get(0, 0)), 'left': int(counts.get(1, 0)),
                    'right': int(counts.get(2, 0))})
display(pd.DataFrame(balance))
preview = CausalIMUFeatures(FEATURES)(train_ds[0]['imu'].unsqueeze(0))
assert torch.isfinite(preview).all()
print('Raw shape:', tuple(train_ds[0]['imu'].shape), '| Engineered shape:', tuple(preview.shape))


  /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_09_10_26/train_data
  /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_09_10_26/valid_data


,split,windows,none,left,right
0,train,2532,1120,696,716
1,valid,1018,440,278,300


Raw shape: (300, 6) | Engineered shape: (1, 300, 20)


## Train a new run
The next cell trains for up to `cfg.training.epochs` epochs. Set it to 1 for a quick smoke experiment. Re-executing this cell starts another independent run in a fresh timestamped directory.

Artifacts: `initial.pt`, `best.pt`, `last.pt`, `history.csv`, `experiment.json`, and `data_manifest.json`. `best.pt` uses the configured validation monitor; `last.pt` and history are updated each completed epoch. The architecture is saved as `model_type='feature_lstm'` with feature settings, channel order, normalization statistics, delay, augmentation, and training seed.

In [20]:
run_dir, history = train_feature_lstm(
    cfg, FEATURES, train_ds, val_ds,
    label_delay_frames=LABEL_DELAY_FRAMES,
    amplitude_augmentation=AMPLITUDE_AUGMENTATION,
)
display(history.tail())
print(f"In evaluate.ipynb set: RUN_DIR = ROOT / {str(run_dir.relative_to(ROOT))!r}")


Run: /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/testing_checkpoints/lstm64_features/20261009_120510_570272_cc56765b
Device: cpu; 20 input features; 48,803 parameters
Epoch   1/60 | train loss=0.5524 acc=0.442 | val loss=0.3914 acc=0.432 P=0.000 R=0.000
Epoch   2/60 | train loss=0.3590 acc=0.442 | val loss=0.3794 acc=0.432 P=0.000 R=0.000
Epoch   3/60 | train loss=0.3359 acc=0.442 | val loss=0.3829 acc=0.432 P=0.000 R=0.000
Epoch   4/60 | train loss=0.3275 acc=0.442 | val loss=0.3956 acc=0.432 P=0.000 R=0.000
Epoch   5/60 | train loss=0.3105 acc=0.539 | val loss=0.3315 acc=0.764 P=0.968 R=0.678
Epoch   6/60 | train loss=0.2255 acc=0.902 | val loss=0.2826 acc=0.871 P=0.987 R=0.794
Epoch   7/60 | train loss=0.2048 acc=0.910 | val loss=0.2325 acc=0.942 P=0.958 R=0.945
Epoch   8/60 | train loss=0.1940 acc=0.924 | val loss=0.2790 acc=0.880 P=0.975 R=0.815
Epoch   9/60 | train loss=0.1852 acc=0.934 | val loss=0.2688 acc=0.898 P=0.956 R=0.863
Epoch  10/60 

,epoch,lr,train_loss,train_frame_acc,train_window_acc,val_loss,val_frame_acc,val_window_acc,val_precision,val_recall,val_tp,val_fp,val_fn,val_tn
20,21,0.000750,0.150291,0.989136,0.982227,0.283717,0.985891,0.944990,0.976321,0.927336,536,13,42,427
21,22,0.000727,0.152551,0.989078,0.979068,0.280558,0.984001,0.948919,0.974865,0.939446,543,14,35,426
22,23,0.000703,0.149381,0.989114,0.977093,0.278514,0.984414,0.951866,0.978378,0.939446,543,12,35,428
23,24,0.000679,0.148880,0.989596,0.977488,0.294195,0.983530,0.942043,0.979742,0.920415,532,11,46,429
24,25,0.000655,0.146766,0.989681,0.983807,0.318472,0.985134,0.925344,0.980916,0.889273,514,10,64,430


In evaluate.ipynb set: RUN_DIR = ROOT / 'testing_checkpoints/lstm64_features/20261009_120510_570272_cc56765b'


## Evaluate and compare
In `evaluate.ipynb`, set `RUN_DIR` to the printed run directory, keep `EVALUATION_DATA_ROOT` on the same prepared split, and use the same `EVALUATION_SEED` and matching rules as the baseline. Restart its kernel once after adding this new architecture so the model factory is current. The factory reconstructs the feature transform and frozen normalization automatically; feed the ordinary six-channel IMU tensors, **not** precomputed features. The overlap-streaming reference includes feature history.

Use the resulting `threshold_comparison/` folder in your comparison notebook. Feature extraction is an architectural difference saved in `model_config`; the evaluation-data fingerprint continues to describe the common raw data and labels. All other data and alarm-policy compatibility checks still apply.

Compare both architectures over the same training seeds. Adding input channels slightly increases projection parameters; there is no assumption that feature engineering will improve accuracy. Window-level training selection remains descriptive validation, not an independent test estimate.